# Synthetic EHR Database — BeakerX SQL / H2

This notebook loads the `synthetic_ehrs.csv` dataset into an **H2 in-memory SQL database**
via the BeakerX SQL kernel and runs exploratory queries against it.

## Schema

```
TABLE ehr_records (
    patient_id  VARCHAR(6),    -- e.g. P0001
    ehr_id      VARCHAR(8),    -- e.g. EHR00000  (one disease per EHR)
    disease     VARCHAR(64),
    symptom     VARCHAR(64)
    -- one row per symptom; an EHR with k symptoms has k rows
)
```

## Contents
1. Connect to H2
2. Create table
3. Load data (54 batched INSERTs, 100 rows each)
4. Verification queries
5. Analytical queries — symptom proportions per disease

In [1]:
%defaultDatasource jdbc:h2:mem:ehr_db

In [2]:
-- ── Create table ─────────────────────────────────────────────────────────────
DROP TABLE IF EXISTS ehr_records;

CREATE TABLE ehr_records (
    patient_id  VARCHAR(6)  NOT NULL,
    ehr_id      VARCHAR(8)  NOT NULL,
    disease     VARCHAR(64) NOT NULL,
    symptom     VARCHAR(64) NOT NULL
) AS SELECT patient_id, ehr_id, disease, symptom
FROM CSVREAD('synthetic_ehrs.csv');

## 4. Verification queries

In [3]:
-- Total rows, EHRs, and patients
SELECT
    COUNT(*)                  AS total_rows,
    COUNT(DISTINCT ehr_id)    AS total_ehrs,
    COUNT(DISTINCT patient_id) AS total_patients,
    COUNT(DISTINCT disease)   AS total_diseases,
    COUNT(DISTINCT symptom)   AS total_symptoms
FROM ehr_records;

In [4]:
-- EHRs and symptom occurrences per disease
SELECT
    disease,
    COUNT(DISTINCT ehr_id)  AS ehrs,
    COUNT(*)                AS symptom_occurrences,
    ROUND(CAST(COUNT(*) AS DOUBLE) / COUNT(DISTINCT ehr_id), 2) AS avg_symptoms_per_ehr
FROM ehr_records
GROUP BY disease
ORDER BY ehrs DESC;

In [5]:
-- EHR distribution per patient (min / avg / max)
SELECT
    MIN(ehr_count)                      AS min_ehrs_per_patient,
    ROUND(AVG(CAST(ehr_count AS DOUBLE)), 2) AS avg_ehrs_per_patient,
    MAX(ehr_count)                      AS max_ehrs_per_patient
FROM (
    SELECT patient_id, COUNT(DISTINCT ehr_id) AS ehr_count
    FROM ehr_records
    GROUP BY patient_id
) t;

In [6]:
-- Sample of 20 records
SELECT * FROM ehr_records LIMIT 20;

## 5. Analytical queries — symptom proportions per disease

In [7]:
-- For each disease, how many EHRs report each symptom?
-- The relative_proportion column should match the importance weights from the source data.
SELECT
    disease,
    symptom,
    COUNT(DISTINCT ehr_id)                                              AS ehrs_with_symptom,
    COUNT(*)                                                            AS occurrences,
    ROUND(
        CAST(COUNT(*) AS DOUBLE)
        / SUM(COUNT(*)) OVER (PARTITION BY disease)
    , 4)                                                                AS relative_proportion
FROM ehr_records
GROUP BY disease, symptom
ORDER BY disease, relative_proportion DESC;

org.h2.jdbc.JdbcSQLException:  Syntax error in SQL statement "SELECT

In [8]:
-- Which symptom is shared by the most diseases?
SELECT
    symptom,
    COUNT(DISTINCT disease) AS diseases_count,
    COUNT(DISTINCT ehr_id)  AS total_ehrs
FROM ehr_records
GROUP BY symptom
ORDER BY diseases_count DESC, total_ehrs DESC;

In [9]:
-- Which patients have EHRs for more than one distinct disease?
SELECT
    patient_id,
    COUNT(DISTINCT disease)  AS distinct_diseases,
    COUNT(DISTINCT ehr_id)   AS total_ehrs,
    GROUP_CONCAT(DISTINCT disease ORDER BY disease) AS diseases
FROM ehr_records
GROUP BY patient_id
HAVING COUNT(DISTINCT disease) > 1
ORDER BY distinct_diseases DESC, total_ehrs DESC
LIMIT 20;

In [10]:
-- Symptom co-occurrence: which symptom pairs appear together most often in the same EHR?
SELECT
    a.symptom   AS symptom_a,
    b.symptom   AS symptom_b,
    COUNT(*)    AS co_occurrences
FROM ehr_records a
JOIN ehr_records b
  ON a.ehr_id = b.ehr_id
 AND a.symptom < b.symptom   -- avoid duplicates and self-joins
GROUP BY a.symptom, b.symptom
ORDER BY co_occurrences DESC
LIMIT 15;